# 5. Integer linear programme

Lecture: *Week 3 - Linear Programming 2*, PDF page 23 (*Integer Solution via Pyomo (problem based)*);
pages 20–22 show the same problem in Desmos.

**The problem.** maximise $P = 2x + 3.5y$ subject to $3x + 2y \le 14$, $2x + 3y \le 12.5$, $-x + y \le 1$,
$x, y \ge 0$, and **$x, y$ whole numbers**. In the code, $x$ is `x[0]` and $y$ is `x[1]`.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** $x = 3$, $y = 2$, $P = 13$. The LP without the whole-number rule has its optimum at
$(1.9, 2.9)$ with $P = 13.95$, and rounding that point does not give $(3, 2)$ (PDF pages 21–22).

*Original script: `5 Source Files/Python/intlinprog_pyomo.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

A *concrete model* is the container for the problem.

In [ ]:
# Create a concrete model
model = ConcreteModel()

The only change from an LP: the domain `NonNegativeIntegers` makes both variables whole numbers $\ge 0$.

In [ ]:
# Integer decision variables x[0] (= x) and x[1] (= y): whole numbers >= 0, no upper bound
model.x = Var([0, 1], domain=NonNegativeIntegers)

Objective coefficients: $P = 2x + 3.5y$.

In [ ]:
# Define the objective function coefficients as a list
f = [2, 3.5]

The objective, maximised.

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(f[i] * model.x[i] for i in [0, 1]), sense=maximize)

The three constraints, first written as Python expressions…

In [ ]:
# Define the inequality constraints
c1 = 3 * model.x[0] + 2 * model.x[1] <= 14    # 3x + 2y <= 14
c2 = 2 * model.x[0] + 3 * model.x[1] <= 12.5  # 2x + 3y <= 12.5
c3 = -model.x[0] + model.x[1] <= 1            # -x + y <= 1

…then attached to the model.

In [ ]:
# Add the constraints to the model
model.c1 = Constraint(expr=c1)
model.c2 = Constraint(expr=c2)
model.c3 = Constraint(expr=c3)

Solve. HiGHS sees the integer variables and uses branch and bound (it solves LPs and splits on fractional values) instead of a single LP.

In [ ]:
# Solve the integer optimization problem
solver = SolverFactory('appsi_highs')
results = solver.solve(model, tee=False)  # Set tee=True to display solver output

If the solver reports *optimal*, print the profit and the two variables.

In [ ]:
# Check the solver status and print the results
if results.solver.status == SolverStatus.ok:
    if results.solver.termination_condition == TerminationCondition.optimal:
        print("Optimal solution found.")
        print("Objective value:", model.obj())
        print("Decision variables:")
        for i in [0, 1]:
            print(f"x[{i}] = {model.x[i].value}")
    else:
        print("Solver terminated with a non-optimal solution.")
else:
    print("Solver did not find a solution. Status:", results.solver.status)
    
    

## Try it

Change `NonNegativeIntegers` to `NonNegativeReals` in the variables cell and re-run all cells. What do you get?

<details><summary>Answer</summary>

$x = 1.9$, $y = 2.9$, $P = 13.95$: the LP optimum from the Desmos slide. The integer answer (13) is lower,
because adding a restriction can never improve the optimum. Change the domain back afterwards.
</details>